In [12]:
BATCH_SIZE = 32
IMG_SIZE = (224, 224)
AUTOTUNE = tf.data.AUTOTUNE

print("Building Training and Validation Datasets...")

all_image_paths = []
all_image_labels = []
class_names = sorted([d.name for d in os.scandir(DATA_DIR) if d.is_dir()])
class_to_idx = {name: i for i, name in enumerate(class_names)}

for class_name in class_names:
    class_dir = os.path.join(DATA_DIR, class_name)
    for img_name in os.listdir(class_dir):
        img_path = os.path.join(class_dir, img_name)
        if os.path.isfile(img_path) and img_name.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp')):
            all_image_paths.append(img_path)
            all_image_labels.append(class_to_idx[class_name])

all_image_paths_tf = tf.constant(all_image_paths, dtype=tf.string)
all_image_labels_tf = tf.constant(all_image_labels, dtype=tf.int32)

dataset = tf.data.Dataset.from_tensor_slices((all_image_paths_tf, all_image_labels_tf))
dataset = dataset.shuffle(buffer_size=len(all_image_paths), seed=123)

dataset_size = len(all_image_paths)
train_size = int(0.8 * dataset_size)

train_ds_raw = dataset.take(train_size)
val_ds_raw = dataset.skip(train_size)

def _decode_and_resize_img(img_path, label, img_size=(224, 224)):
    def _decode_image_py_func(path, lbl):
        try:
            img_bytes = tf.io.read_file(path)
            img = tf.image.decode_image(img_bytes, channels=3, expand_animations=False)
            img = tf.image.resize(img, img_size)
            img = tf.cast(img, tf.uint8)
            return img, lbl, True
        except Exception as e:

            return tf.zeros((img_size[0], img_size[1], 3), dtype=tf.uint8), lbl, False

    image, label, success = tf.py_function(
        _decode_image_py_func,
        inp=[img_path, label],
        Tout=[tf.uint8, tf.int32, tf.bool]
    )

    image.set_shape([img_size[0], img_size[1], 3])
    label.set_shape([])
    success.set_shape([])

    return image, label, success

def configure_for_performance(ds, shuffle_data=False):
    ds = ds.map(lambda img_path, label: _decode_and_resize_img(img_path, label, IMG_SIZE), num_parallel_calls=AUTOTUNE)
    ds = ds.filter(lambda image, label, success: success)
    ds = ds.map(lambda image, label, success: (image, label))

    if shuffle_data:
        ds = ds.shuffle(buffer_size=1000)
    ds = ds.batch(BATCH_SIZE)
    ds = ds.cache()
    ds = ds.prefetch(buffer_size=AUTOTUNE)
    return ds

train_dataset = configure_for_performance(train_ds_raw, shuffle_data=True)
val_dataset = configure_for_performance(val_ds_raw)

print(f"\nConstructors successfully identified ({len(class_names)} classes):")
print(class_names)

Building Training and Validation Datasets...

Constructors successfully identified (8 classes):
['AlphaTauri F1 car', 'Ferrari F1 car', 'McLaren F1 car', 'Mercedes F1 car', 'Racing Point F1 car', 'Red Bull Racing F1 car', 'Renault F1 car', 'Williams F1 car']


In [13]:
import tensorflow as tf
from tensorflow.keras import layers, models

num_classes = len(class_names)

print(f"Constructing Baseline CNN for {num_classes} classes...")

model1 = models.Sequential([

    layers.Rescaling(1./255, input_shape=(224, 224, 3)),

    layers.Conv2D(32, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(64, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Conv2D(128, (3, 3), activation='relu'),
    layers.MaxPooling2D((2, 2)),

    layers.Flatten(),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),

    layers.Dense(num_classes, activation='softmax')
])

model1.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model1.summary()


Constructing Baseline CNN for 8 classes...


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ rescaling_1 (Rescaling)         │ (None, 224, 224, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 222, 222, 32)   │           896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_3 (MaxPooling2D)  │ (None, 111, 111, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 109, 109, 64)   │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_4 (MaxPooling2D)  │ (None, 54, 54, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 52, 52, 128)    │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_5 (MaxPooling2D)  │ (None, 26, 26, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten_1 (Flatten)             │ (None, 86528)          │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │    11,075,712 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 8)              │         1,032 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 11,169,992 (42.61 MB)

 Trainable params: 11,169,992 (42.61 MB)

 Non-trainable params: 0 (0.00 B)